In [1]:
import sys
print(sys.executable)

C:\Users\Pavan Chand\anaconda3\envs\rag_quiz\python.exe


In [59]:
import os
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

pdf_folder = "data"

for file in os.listdir(pdf_folder):

    if file.endswith(".pdf"):

        pdf_path = os.path.join(pdf_folder, file)

        loader = PyPDFLoader(pdf_path)

        docs = loader.load()

        all_docs.extend(docs)

print("Total Pages:", len(all_docs))

Total Pages: 6


#Split pdf into text


In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [61]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])


Number of chunks: 11
Analysis vs Analytics 
Alright! So… 
Let’s discuss the not-so-obvious differences 
between the terms analysis and analytics. 
Due to the similarity of the words, some people 
believe they share the same meaning, and thus 
use them interchangeably. Technically, this 
isn’t correct. There is, in fact, a distinct 
difference between the two. And the reason 
for one often being used instead of the other 
is the lack of a transparent understanding 
of both. 
So, let’s clear this up, shall we? 
First,


In [62]:
#create Embeddings

In [63]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [64]:
#Create Chroma Vector Database
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db"
)

print("Vector DB created")

Vector DB created


In [65]:

#Test Retrieval
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke("important concepts")

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:500])


--- Chunk 1 ---
they relate to other parts. And that’s analysis 
in a nutshell. 
One important thing to remember, however, 
is that you perform analyses on things that 
have already happened in the past. Such as 
using an analysis to explain how a story ended 
the way it did or how there was a decrease 
in sales last summer. 
All this means that we do analyses to explain 
how and/or why something happened. 
Great! 
Now, this leads us nicely on to the definition 
of analytics. 
As you have probably guessed, anal

--- Chunk 2 ---
they relate to other parts. And that’s analysis 
in a nutshell. 
One important thing to remember, however, 
is that you perform analyses on things that 
have already happened in the past. Such as 
using an analysis to explain how a story ended 
the way it did or how there was a decrease 
in sales last summer. 
All this means that we do analyses to explain 
how and/or why something happened. 
Great! 
Now, this leads us nicely on to the definition 
of analytics. 

In [66]:
import sys
print(sys.executable)

C:\Users\Pavan Chand\anaconda3\envs\rag_quiz\python.exe


In [67]:
import langchain_ollama
print("Installed successfully")

Installed successfully


In [68]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0.3
)

response = llm.invoke("What is Python in one sentence?")
print(response.content)

Python is a high-level, interpreted programming language that is widely used for various applications such as web development, data analysis, artificial intelligence, and more due to its simplicity, readability, and versatility.


In [69]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [70]:
results = retriever.invoke("important concepts")

In [71]:
context = "\n\n".join([doc.page_content for doc in results])

In [72]:
prompt = f"""
Generate 10 multiple choice questions.

IMPORTANT:
- Each question MUST have exactly 4 options.
- Options must be meaningful answer choices.
- Do NOT use just "A", "B", "C", or "D" as option text.
- Return ONLY valid JSON.

Example:

[
  {{
    "question":"What is Python?",
    "options":[
      "Programming language",
      "Database",
      "Operating system",
      "Web browser"
    ],
    "answer":"A"
  }}
]

Content:
{context}
"""

response = llm.invoke(prompt)

print(response.content)

[
  {
    "question": "What do analyses typically relate to?",
    "options": [
      "Other past events",
      "Component parts obtained in an analysis",
      "Future potential outcomes",
      "Logical and computational reasoning"
    ],
    "answer": "A"
  },
  {
    "question": "When performing analyses, what is being explained?",
    "options": [
      "How something will happen in the future",
      "Why something happened in the past",
      "How and/or why something happened",
      "What something means"
    ],
    "answer": "C"
  },
  {
    "question": "What does analytics generally refer to?",
    "options": [
      "Explaining past events",
      "Exploring potential future outcomes",
      "Applying logical and computational reasoning",
      "Analyzing component parts"
    ],
    "answer": "B"
  },
  {
    "question": "What is the primary goal of analytics?",
    "options": [
      "Explaining how something happened",
      "Exploring patterns in data",
      "Identifyi

In [73]:
#parse jason
raw_text = response.content

match = re.search(
    r"```(?:json)?\s*(.*?)\s*```",
    raw_text,
    re.DOTALL
)

if match:
    json_text = match.group(1)
else:
    start = raw_text.find("[")
    end = raw_text.rfind("]") + 1
    json_text = raw_text[start:end]

quiz_data = json.loads(json_text)

print(type(quiz_data))
print(quiz_data[0])

<class 'list'>
{'question': 'What do analyses typically relate to?', 'options': ['Other past events', 'Component parts obtained in an analysis', 'Future potential outcomes', 'Logical and computational reasoning'], 'answer': 'A'}


In [74]:
#Display Quiz
for i, q in enumerate(quiz_data, start=1):

    print(f"\nQuestion {i}")
    print(q["question"])

    for idx, option in enumerate(q["options"]):
        print(f"{chr(65+idx)}. {option}")

    print("Correct Answer:", q["answer"])


Question 1
What do analyses typically relate to?
A. Other past events
B. Component parts obtained in an analysis
C. Future potential outcomes
D. Logical and computational reasoning
Correct Answer: A

Question 2
When performing analyses, what is being explained?
A. How something will happen in the future
B. Why something happened in the past
C. How and/or why something happened
D. What something means
Correct Answer: C

Question 3
What does analytics generally refer to?
A. Explaining past events
B. Exploring potential future outcomes
C. Applying logical and computational reasoning
D. Analyzing component parts
Correct Answer: B

Question 4
What is the primary goal of analytics?
A. Explaining how something happened
B. Exploring patterns in data
C. Identifying potential future outcomes
D. Applying logical and computational reasoning
Correct Answer: C

Question 5
When performing analyses, what is being looked for?
A. Patterns in the past
B. Logical and computational reasoning
C. Future pot

In [83]:
#run quiz
score = 0
user_answers = []

for i, q in enumerate(quiz_data, start=1):

    print(f"\nQuestion {i}")
    print(q["question"])

    for idx, option in enumerate(q["options"]):
        print(f"{chr(65+idx)}. {option}")

    valid_choices = [
        chr(65+i)
        for i in range(len(q["options"]))
    ]

    user_answer = input(
        f"Your answer ({'/'.join(valid_choices)}): "
    ).strip().upper()

    user_answers.append(user_answer)

    if user_answer == q["answer"]:
        score += 1

print(f"\nFinal Score: {score}/{len(quiz_data)}")


Question 1
What do analyses typically relate to?
A. Other past events
B. Component parts obtained in an analysis
C. Future potential outcomes
D. Logical and computational reasoning


Your answer (A/B/C/D):  a



Question 2
When performing analyses, what is being explained?
A. How something will happen in the future
B. Why something happened in the past
C. How and/or why something happened
D. What something means


Your answer (A/B/C/D):  a



Question 3
What does analytics generally refer to?
A. Explaining past events
B. Exploring potential future outcomes
C. Applying logical and computational reasoning
D. Analyzing component parts


...


Question 9
When performing analyses, what is being looked for?
A. Patterns in the past
B. Logical and computational reasoning
C. Future potential outcomes
D. Component parts obtained in an analysis


Your answer (A/B/C/D):  a



Question 10
What does analytics typically involve?
A. Explaining how something happened
B. Exploring patterns in data
C. Identifying potential future outcomes
D. Applying logical and computational reasoning


Your answer (A/B/C/D):  a



Final Score: 4/10


In [87]:
#Show Wrong Answers
for q, user_ans in zip(
    quiz_data,
    user_answers
):

    if user_ans != q["answer"]:

        print("\n--------------------")
        print("Question:", q["question"])
        print("Your Answer:", user_ans)
        print("Correct Answer:", q["answer"])


--------------------
Question: When performing analyses, what is being explained?
Your Answer: A
Correct Answer: C

--------------------
Question: What does analytics generally refer to?
Your Answer: A
Correct Answer: B

--------------------
Question: What is the primary goal of analytics?
Your Answer: A
Correct Answer: C

--------------------
Question: What do analyses typically focus on?
Your Answer: A
Correct Answer: B

--------------------
Question: What does analytics aim to achieve?
Your Answer: A
Correct Answer: D

--------------------
Question: When performing analyses, what is being looked for?
Your Answer: A
Correct Answer: C


In [88]:
#AI explainations

for q, user_ans in zip(
    quiz_data,
    user_answers
):

    if user_ans != q["answer"]:

        prompt = f"""
        Question:
        {q['question']}

        Correct Answer:
        {q['answer']}

        Explain why this answer is correct
        using the source content.
        """

        explanation = llm.invoke(prompt)

        print("\n================")
        print(q["question"])
        print(explanation.content)


When performing analyses, what is being explained?
The question is asking about the explanation of "C" in a context that isn't provided. However, I can provide an example based on common data analysis contexts.

In data analysis, "C" could refer to a correlation coefficient or a confidence interval. 

For instance, if we're discussing regression analysis, "C" might represent the coefficient of determination (R-squared), which measures how well the model fits the data. In this case:

- A = The independent variable(s)
- B = The dependent variable
- C = The R-squared value

In this scenario, "C" is being explained as a measure of the goodness of fit for the regression model.

However, without more context, it's impossible to provide a definitive answer.

What does analytics generally refer to?
The question asks what "analytics" generally refers to.

According to the provided source content, analytics can be defined as:

"Analytics: The process of examining data sets to draw conclusions a

In [89]:
#save quiz
with open(
    "quiz.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        quiz_data,
        f,
        indent=4
    )

print("Quiz Saved")

Quiz Saved


In [90]:
import os

print(os.path.exists("quiz.json"))


True


In [91]:
import json

with open("quiz.json", "r", encoding="utf-8") as f:
    quiz_data = json.load(f)

print(len(quiz_data))

10


In [92]:
import streamlit as st

st.title("PDF Quiz Generator")

uploaded_file = st.file_uploader(
    "Upload PDF",
    type="pdf"
)

if uploaded_file:
    st.success("PDF Uploaded Successfully")

2026-06-22 10:28:36.767 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.024 
  command:

    streamlit run C:\Users\Pavan Chand\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-22 10:28:37.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.026 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-22 10:28:37.03

In [93]:
load_pdf()
split_text()
create_embeddings()
retrieve_context()
generate_quiz()

NameError: name 'load_pdf' is not defined